# 🧠 GoogLeNet(InceptionV3) & ResNet50 실습 (Keras Applications)

이 노트북에서는 Keras Applications을 활용하여 GoogLeNet(InceptionV3)과 ResNet50을 실습합니다.

## 주요 내용
1. 사전학습(Pretrained) 모델 불러오기
2. 모델 튜닝 (Fine-tuning)
3. 전이학습 (Transfer Learning)
4. 파라미터 수 비교

In [1]:
import tensorflow as tf
from tensorflow.keras.applications import InceptionV3, ResNet50
from tensorflow.keras.applications.inception_v3 import preprocess_input, decode_predictions
from tensorflow.keras.preprocessing import image
from tensorflow.keras import layers, models, optimizers
import numpy as np
print('TensorFlow version:', tf.__version__)

I0000 00:00:1791888363.514719 3431161 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1791888363.553378 3431161 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1791888364.521884 3431161 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


TensorFlow version: 2.21.0


## [1] GoogLeNet (InceptionV3) 모델 불러오기

In [2]:
inception = InceptionV3(weights='imagenet', include_top=True)
inception.summary()

W0000 00:00:1791888365.758333 3431161 gpu_device.cc:2459] TensorFlow was not built with CUDA kernel binaries compatible with compute capability 12.0a. CUDA kernels will be jit-compiled from PTX, which could take 30 minutes or longer.
W0000 00:00:1791888365.765400 3431161 gpu_device.cc:2459] TensorFlow was not built with CUDA kernel binaries compatible with compute capability 12.0a. CUDA kernels will be jit-compiled from PTX, which could take 30 minutes or longer.
I0000 00:00:1791888365.854998 3431161 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 11156 MB memory:  -> device: 0, name: NVIDIA RTX PRO 4000 Blackwell, pci bus id: 0000:01:00.0, compute capability: 12.0a


Model: "inception_v3"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 299, 299,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d (Conv2D)     │ (None, 149, 149,  │        864 │ input_layer[0][0] │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 149, 149,  │         96 │ conv2d[0][0]      │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation          │ (None, 149, 149,  │          0 │ batch_normalizat… │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_1 (Conv2D)   │ (None, 147, 147,  │      9,216 │ activation[0][0]  │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 147, 147,  │         96 │ conv2d_1[0][0]    │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_1        │ (None, 147, 147,  │          0 │ batch_normalizat… │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_2 (Conv2D)   │ (None, 147, 147,  │     18,432 │ activation_1[0][… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 147, 147,  │        192 │ conv2d_2[0][0]    │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_2        │ (None, 147, 147,  │          0 │ batch_normalizat… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d       │ (None, 73, 73,    │          0 │ activation_2[0][… │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_3 (Conv2D)   │ (None, 73, 73,    │      5,120 │ max_pooling2d[0]… │
│                     │ 80)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 73, 73,    │        240 │ conv2d_3[0][0]    │
│ (BatchNormalizatio… │ 80)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_3        │ (None, 73, 73,    │          0 │ batch_normalizat… │
│ (Activation)        │ 80)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_4 (Conv2D)   │ (None, 71, 71,    │    138,240 │ activation_3[0][… │
│                     │ 192)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 71, 71,    │        576 │ conv2d_4[0][0]    │
│ (BatchNormalizatio… │ 192)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_4        │ (None, 71, 71,    │          0 │ batch_normalizat

 Total params: 23,851,784 (90.99 MB)

 Trainable params: 23,817,352 (90.86 MB)

 Non-trainable params: 34,432 (134.50 KB)

### 예시 이미지로 예측

In [3]:
img_path = tf.keras.utils.get_file('elephant.jpg', 'https://storage.googleapis.com/download.tensorflow.org/example_images/elephant.jpg')
img = image.load_img(img_path, target_size=(299, 299))
x = image.img_to_array(img)
x = np.expand_dims(x, axis=0)
x = preprocess_input(x)
preds = inception.predict(x)
print('예측 결과:', decode_predictions(preds, top=3)[0])

I0000 00:00:1791888368.899035 3431383 service.cc:153] XLA service 0x7436b006e6a0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1791888368.899052 3431383 service.cc:161]   StreamExecutor [0]: NVIDIA RTX PRO 4000 Blackwell, Compute Capability 12.0a (Driver: 13.0.0; Runtime: 12.9.0; Toolkit: 12.5.0; DNN: 9.19.0)
I0000 00:00:1791888368.949130 3431383 dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1791888369.607181 3431383 cuda_dnn.cc:461] Loaded cuDNN version 91900


1/1 ━━━━━━━━━━━━━━━━━━━━ 8s 8s/step
예측 결과: [('n02504458', 'African_elephant', np.float32(0.71341217)), ('n01871265', 'tusker', np.float32(0.14513122)), ('n02504013', 'Indian_elephant', np.float32(0.038662553))]


I0000 00:00:1791888375.683659 3431383 device_compiler.h:208] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


## [2] ResNet50 모델 불러오기

In [4]:
resnet = ResNet50(weights='imagenet', include_top=True)
resnet.summary()

Model: "resnet50"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1       │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_pad           │ (None, 230, 230,  │          0 │ input_layer_1[0]… │
│ (ZeroPadding2D)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_conv (Conv2D) │ (None, 112, 112,  │      9,472 │ conv1_pad[0][0]   │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_bn            │ (None, 112, 112,  │        256 │ conv1_conv[0][0]  │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_relu          │ (None, 112, 112,  │          0 │ conv1_bn[0][0]    │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool1_pad           │ (None, 114, 114,  │          0 │ conv1_relu[0][0]  │
│ (ZeroPadding2D)     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool1_pool          │ (None, 56, 56,    │          0 │ pool1_pad[0][0]   │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_conv │ (None, 56, 56,    │      4,160 │ pool1_pool[0][0]  │
│ (Conv2D)            │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_bn   │ (None, 56, 56,    │        256 │ conv2_block1_1_c… │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_relu │ (None, 56, 56,    │          0 │ conv2_block1_1_b… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_conv │ (None, 56, 56,    │     36,928 │ conv2_block1_1_r… │
│ (Conv2D)            │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_bn   │ (None, 56, 56,    │        256 │ conv2_block1_2_c… │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_relu │ (None, 56, 56,    │          0 │ conv2_block1_2_b… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_0_conv │ (None, 56, 56,    │     16,640 │ pool1_pool[0][0]  │
│ (Conv2D)            │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_3_conv │ (None, 56, 56,    │     16,640 │ conv2_block1_2_r… │
│ (Conv2D)            │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_0_bn   │ (None, 56, 56,    │      1,024 │ conv2_block1_0_c… │
│ (BatchNormalizatio… │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_3_bn   │ (None, 56, 56,    │      1,024 │ conv2_block1_3_c

 Total params: 25,636,712 (97.80 MB)

 Trainable params: 25,583,592 (97.59 MB)

 Non-trainable params: 53,120 (207.50 KB)

In [5]:
img_res = image.load_img(img_path, target_size=(224, 224))
x_res = image.img_to_array(img_res)
x_res = np.expand_dims(x_res, axis=0)
x_res = tf.keras.applications.resnet50.preprocess_input(x_res)
preds_res = resnet.predict(x_res)
print('ResNet50 예측 결과:', tf.keras.applications.resnet50.decode_predictions(preds_res, top=3)[0])

1/1 ━━━━━━━━━━━━━━━━━━━━ 4s 4s/step
ResNet50 예측 결과: [('n02504458', 'African_elephant', np.float32(0.65602446)), ('n01871265', 'tusker', np.float32(0.24329455)), ('n02504013', 'Indian_elephant', np.float32(0.10037672))]


## [3] 모델 튜닝 (Fine-tuning)

In [6]:
base_model = InceptionV3(weights='imagenet', include_top=False, input_shape=(150,150,3))
base_model.trainable = False

model_ft = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(5, activation='softmax')
])

model_ft.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model_ft.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ inception_v3 (Functional)       │ (None, 3, 3, 2048)     │    21,802,784 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 2048)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       262,272 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 5)              │           645 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 22,065,701 (84.17 MB)

 Trainable params: 262,917 (1.00 MB)

 Non-trainable params: 21,802,784 (83.17 MB)

In [7]:
for layer in base_model.layers[-30:]:
    layer.trainable = True
model_ft.compile(optimizer=tf.keras.optimizers.Adam(1e-5), loss='categorical_crossentropy', metrics=['accuracy'])

## [4] 전이학습 (Transfer Learning)

In [8]:
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks, optimizers
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input
from tensorflow.keras.datasets import cifar10
from tensorflow.keras.utils import to_categorical

(x_train, y_train), (x_test, y_test) = cifar10.load_data()
x_train = x_train.astype('float32')   # /255 하지 않음 (0~255 유지)
x_test  = x_test.astype('float32')
y_train = to_categorical(y_train, 10)
y_test  = to_categorical(y_test, 10)

IMG = 160  # GPU 메모리에 따라 96~224

feature_model = ResNet50(weights='imagenet', include_top=False,
                         input_shape=(IMG, IMG, 3))
feature_model.trainable = False

inputs = layers.Input(shape=(32, 32, 3))
x = layers.Resizing(IMG, IMG)(inputs)
x = layers.Lambda(preprocess_input)(x)
x = feature_model(x, training=False)      # BN을 추론 모드로 고정
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(256, activation='relu')(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(10, activation='softmax')(x)
transfer_model = models.Model(inputs, outputs)


transfer_model.compile(optimizer=optimizers.Adam(1e-5),
                       loss='categorical_crossentropy', metrics=['accuracy'])

transfer_model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_5 (InputLayer)      │ (None, 32, 32, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ resizing (Resizing)             │ (None, 160, 160, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lambda (Lambda)                 │ (None, 160, 160, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ resnet50 (Functional)           │ (None, 5, 5, 2048)     │    23,587,712 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_1      │ (None, 2048)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 256)            │       524,544 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 10)             │         2,570 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 24,114,826 (91.99 MB)

 Trainable params: 527,114 (2.01 MB)

 Non-trainable params: 23,587,712 (89.98 MB)

In [9]:
transfer_model.fit(x_train, y_train, epochs=100, batch_size=128,
                   validation_split=0.1, verbose=2)

test_loss, test_acc = transfer_model.evaluate(x_test, y_test, verbose=0)
print(f'전이학습 테스트 정확도: {test_acc:.4f}')

Epoch 1/100


I0000 00:00:1791888388.765364 3431381 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_26129__.188
I0000 00:00:1791888404.341494 3431381 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_26129__.188


352/352 - 29s - 82ms/step - accuracy: 0.4124 - loss: 1.8236 - val_accuracy: 0.7540 - val_loss: 0.8239
Epoch 2/100
352/352 - 13s - 36ms/step - accuracy: 0.7106 - loss: 0.8727 - val_accuracy: 0.8244 - val_loss: 0.5547
Epoch 3/100
352/352 - 13s - 37ms/step - accuracy: 0.7801 - loss: 0.6663 - val_accuracy: 0.8474 - val_loss: 0.4696
Epoch 4/100
352/352 - 13s - 37ms/step - accuracy: 0.8056 - loss: 0.5737 - val_accuracy: 0.8560 - val_loss: 0.4266
Epoch 5/100
352/352 - 13s - 37ms/step - accuracy: 0.8235 - loss: 0.5178 - val_accuracy: 0.8614 - val_loss: 0.4017
Epoch 6/100
352/352 - 13s - 37ms/step - accuracy: 0.8357 - loss: 0.4792 - val_accuracy: 0.8650 - val_loss: 0.3842
Epoch 7/100
352/352 - 13s - 37ms/step - accuracy: 0.8479 - loss: 0.4535 - val_accuracy: 0.8702 - val_loss: 0.3711
Epoch 8/100
352/352 - 13s - 37ms/step - accuracy: 0.8535 - loss: 0.4308 - val_accuracy: 0.8744 - val_loss: 0.3606
Epoch 9/100
352/352 - 13s - 38ms/step - accuracy: 0.8602 - loss: 0.4102 - val_accuracy: 0.8752 - val

In [10]:
models_info = {
    'InceptionV3': inception.count_params(),
    'ResNet50': resnet.count_params(),
    'Fine-tuned Inception': model_ft.count_params(),
    'Transfer ResNet50': transfer_model.count_params()
}
for name, params in models_info.items():
    print(f'{name}: {params:,} parameters')

InceptionV3: 23,851,784 parameters
ResNet50: 25,636,712 parameters
Fine-tuned Inception: 22,065,701 parameters
Transfer ResNet50: 24,114,826 parameters
